# 01 - Data Exploration

**Goal:** load the mutation data, look at it, and decide what the label can honestly mean.

Order of work in this notebook:
1. Load raw MAF files (or synthetic data if you have not downloaded anything yet)
2. VAF distribution per cohort - this is where the real signal lives
3. Gene frequency per cohort - **this is where the leakage lives**
4. Decide the label definition

> If you take one thing from this notebook: check step 3 before you train anything.
> If a gene appears in the AML file and never in the CHIP file, the model will learn
> that gene, get AUC 0.95, and tell you nothing about early detection.

In [11]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/leukemia-prediagnose/blood-cancer-early-detection")
    # Run every session, not only on first mount: Colab wipes packages on
    # restart, and misspelled package names fail silently without check=True.
    subprocess.run("pip install -q transformers accelerate lightgbm".split(), check=True)
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)

CFG = yaml.safe_load(open("configs/config.yaml"))
CFG

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
working dir: /content/drive/MyDrive/leukemia-prediagnose/blood-cancer-early-detection


{'project': {'name': 'blood-cancer-early-detection', 'seed': 42},
 'data': {'raw_dir': 'data/raw',
  'processed_dir': 'data/processed',
  'external_dir': 'data/external',
  'maf_file': 'data/raw/data_mutations_extended.txt',
  'clinical_file': 'data/raw/data_clinical_patient.txt',
  'uniprot_cache': 'data/external/uniprot_cache.json'},
 'filters': {'min_vaf': 0.02,
  'min_depth': 100,
  'keep_kinds': ['missense', 'nonsense', 'frameshift'],
  'use_ch_panel': True},
 'labels': {'strategy': 'longitudinal',
  'outcome_col': 'progressed_to_aml',
  'positive_diagnoses': ['AML', 'MDS', 'MPN', 'CMML']},
 'embedding': {'model_name': 'facebook/esm2_t33_650M_UR50D',
  'batch_size': 4,
  'weight_by_vaf': True},
 'model': {'kind': 'logreg',
  'n_pca': 32,
  'n_splits': 5,
  'target_specificity': 0.98,
  'out_path': 'models/classifier.joblib'}}

## 1. Load

Download from cBioPortal and unzip into `data/raw/`:
- **CHIP (negatives):** MSK-IMPACT clonal haematopoiesis / Bolton 2020
- **AML (positives):** `laml_tcga` and/or BeatAML (OHSU)

Both give you `data_mutations_extended.txt`.

Tag every source with `cohort=` when you load it. Step 3 cannot run without it,
and merging two studies untagged is how this project fails silently.

`cohort_confound=` deliberately simulates the broken design (cases from one
study, controls from another, disjoint panels). Set it to 0.9 once and watch
step 3 fire - that is how you confirm the detector works before trusting a
clean report on real data.

In [12]:
from src.data_processing import load_maf, clean_mutations, aggregate_by_patient
from src.synthetic import make_synthetic_cohort
from src.mutation_utils import CH_GENES

USE_SYNTHETIC = not os.path.exists(CFG["data"]["maf_file"])

if USE_SYNTHETIC:
    print(">>> Real MAF not found. Using SYNTHETIC data. Results are meaningless.")
    maf, clinical, SYN_SEQS = make_synthetic_cohort(
        n_patients=600,
        difficulty=0.65,       # 1.0 = no signal at all, use as a null control
        cohort_confound=0.0,   # try 0.9 to simulate the broken study design
        return_sequences=True,
    )
    # Persist them: synthetic mutations only make sense against synthetic
    # sequences. Checked against real UniProt they match ~5% of the time,
    # which is chance for 20 amino acids, and notebook 02 will (correctly)
    # flag a mismatch that is an artefact of mixing the two sources.
    import json as _json
    os.makedirs(CFG["data"]["processed_dir"], exist_ok=True)
    with open(f'{CFG["data"]["processed_dir"]}/synthetic_sequences.json', "w") as f:
        _json.dump(SYN_SEQS, f)
else:
    frames = [load_maf(s["path"], columns=s.get("columns"), cohort=s["cohort"])
              for s in CFG["data"]["sources"]]
    maf = pd.concat(frames, ignore_index=True)
    clinical = load_maf(CFG["data"]["clinical_file"])

print(maf.shape)
maf.head()

>>> Real MAF not found. Using SYNTHETIC data. Results are meaningless.
[synthetic] 1,285 mutations, 600 patients, 16.0% positive, difficulty=0.65, cohort_confound=0.0
(1285, 7)


,Hugo_Symbol,HGVSp_Short,Variant_Classification,Tumor_Sample_Barcode,t_alt_count,t_ref_count,cohort
0,U2AF1,p.C183R,Missense_Mutation,PT-0000-01,131,855,SYNTHETIC
1,TP53,p.C307Q,Missense_Mutation,PT-0000-01,344,743,SYNTHETIC
2,STAG2,p.E80V,Missense_Mutation,PT-0000-01,191,836,SYNTHETIC
3,NF1,p.A1552E,Missense_Mutation,PT-0001-01,242,863,SYNTHETIC
4,CEBPA,p.G298P,Missense_Mutation,PT-0002-01,215,685,SYNTHETIC


## 2. Clean + VAF

`clean_mutations` standardises column names, computes VAF from allele counts,
restricts to the CH gene panel, parses HGVSp notation, and drops variants below
the VAF and depth floors.

Depth matters more than it looks: VAF 0.02 at depth 100 is **2 reads**, which is
sequencing noise, not a clone. If cases and controls were sequenced at different
depths, this filter is what stops the model learning the assay.

In [ ]:
mut = clean_mutations(
    maf,
    genes=CH_GENES if CFG["filters"]["use_ch_panel"] else None,
    min_vaf=CFG["filters"]["min_vaf"],
    min_depth=CFG["filters"].get("min_depth", 0),
    keep_kinds=CFG["filters"]["keep_kinds"],
)
mut.head()

In [ ]:
patients = aggregate_by_patient(mut)
labels = patients.merge(
    clinical[["patient_id", CFG["labels"]["outcome_col"]]],
    on="patient_id", how="inner",
).rename(columns={CFG["labels"]["outcome_col"]: "label"})

print(f"{len(labels):,} patients | {int(labels.label.sum()):,} positive "
      f"({labels.label.mean():.1%})")
labels[["patient_id", "n_mutations", "max_vaf", "n_genes", "label"]].head()

## 3. VAF distribution per class

What you want: the two classes overlap heavily, with positives shifted toward
higher VAF. That shift is real biology - an expanding clone is a clone on its
way to becoming a disease.

What you do **not** want: non-overlapping distributions, or different left-hand
cut-offs. A different VAF floor between classes means a different detection
limit, i.e. a different assay, i.e. a confound.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

vl = mut.merge(labels[["patient_id", "label"]], on="patient_id")
for lb, name in [(0, "no progression"), (1, "progressed")]:
    ax[0].hist(vl.loc[vl.label == lb, "vaf"], bins=40, alpha=.55,
               density=True, label=name)
ax[0].set_xlabel("VAF"); ax[0].set_ylabel("density")
ax[0].set_title("VAF by class (want: overlap + shift)"); ax[0].legend()

labels.boxplot(column="max_vaf", by="label", ax=ax[1])
ax[1].set_title("max VAF per patient"); ax[1].set_xlabel("label")

labels.boxplot(column="n_mutations", by="label", ax=ax[2])
ax[2].set_title("mutations per patient"); ax[2].set_xlabel("label")

plt.suptitle(""); plt.tight_layout(); plt.show()

In [ ]:
# Single-feature AUCs. This is the bar ESM2 has to clear in notebook 04.
from sklearn.metrics import roc_auc_score
y = labels["label"].to_numpy()
for f in ["max_vaf", "sum_vaf", "n_mutations", "mean_vaf", "n_genes"]:
    print(f"{f:>14s}  AUC = {roc_auc_score(y, labels[f]):.3f}")

## 4. Gene frequency per cohort - the leakage check

This is the cell that decides whether the project is real.

1. **Cohort x label.** If one cohort is nearly all positives and the other
   nearly all negatives, cohort *is* the label, and every file-level difference
   between the two studies becomes a free predictor.
2. **Gene exclusivity.** A gene appearing in one class and never the other is
   almost always a panel difference, not biology. Real CH genes appear in both -
   DNMT3A is the commonest CH mutation *and* common in AML.
3. **Assay depth.** Different median depth between classes means different
   detection limits.

If any of these fire, fix the data. No model change rescues a confounded design.

In [ ]:
from src.diagnostics import leakage_report, shared_gene_panel

report = leakage_report(mut, labels)

In [ ]:
# Genes usable by both classes. Train on these, not on the full panel.
panel = shared_gene_panel(mut, labels, min_patients=3)
print(panel[:25])

dropped = sorted(set(mut.gene.unique()) - set(panel))
if dropped:
    print(f"\ndropped {len(dropped)}: {dropped[:20]}")

In [ ]:
gf = report["gene_freq"].head(20)
gf[["freq_0", "freq_1"]].plot.barh(figsize=(8, 7))
plt.xlabel("fraction of patients in class"); plt.gca().invert_yaxis()
plt.title("Gene frequency by class (sorted by exclusivity)")
plt.tight_layout(); plt.show()

## 5. Decide the label definition

Write down which one you are using, because it determines what the number in
notebook 04 is allowed to claim.

| definition | claim it supports | honest? |
|---|---|---|
| progressed to AML/MDS at follow-up | early detection | **yes** |
| sample is from an AML study | AML vs CH classification | no - label is a function of the features |
| high-risk genotype rule | reproducing a published rule | no - circular |

The first requires longitudinal data: the same person sampled while still
healthy, outcome recorded years later. Public sources are scarce and mostly
behind dbGaP (Abelson 2018 Nature, Desai 2018 Nat Med, Bolton 2020 Nat Genet).
Getting access is the critical path for this project. The code is not - it is
already written.

If checks 1-3 came back clean and you have a real outcome column, go to
notebook 02. If not, fix the cohort composition first.

In [ ]:
os.makedirs(CFG["data"]["processed_dir"], exist_ok=True)
mut.to_parquet(f'{CFG["data"]["processed_dir"]}/mutations_clean.parquet')
clinical.to_parquet(f'{CFG["data"]["processed_dir"]}/clinical.parquet')
pd.Series(panel, name="gene").to_csv(
    f'{CFG["data"]["processed_dir"]}/shared_panel.csv', index=False)
print("saved: mutations_clean.parquet, clinical.parquet, shared_panel.csv")